In [2]:
import os
import shutil
import random

# ---- PATHS ----
images_dir = "E:/Hirusha/data/volley-im-new/Implementation-volley/img/Train"
annotations_dir = "E:/Hirusha/data/volley-im-new/Implementation-volley/img/Ann_data"

output_train_images = "dataset/train/images"
output_train_ann = "dataset/train/annotations"
output_val_images = "dataset/val/images"
output_val_ann = "dataset/val/annotations"

split_ratio = 0.8  # 80% train, 20% validation

# ---- Create output folders ----
os.makedirs(output_train_images, exist_ok=True)
os.makedirs(output_train_ann, exist_ok=True)
os.makedirs(output_val_images, exist_ok=True)
os.makedirs(output_val_ann, exist_ok=True)

# ---- Get list of image files ----
images = [f for f in os.listdir(images_dir) 
          if f.lower().endswith((".jpg", ".jpeg", ".png"))]

# Shuffle before splitting
random.shuffle(images)

# Train/val split
split_index = int(len(images) * split_ratio)
train_files = images[:split_index]
val_files = images[split_index:]

def copy_pairs(file_list, img_out, ann_out):
    for img_file in file_list:
        img_path = os.path.join(images_dir, img_file)

        # annotation must match image name without extension
        base = os.path.splitext(img_file)[0]
        
        # find annotation file
        ann_file = None
        for ext in [".xml", ".txt", ".json"]:
            if os.path.exists(os.path.join(annotations_dir, base + ext)):
                ann_file = base + ext
                break

        if ann_file is None:
            print(f"Warning: No annotation found for {img_file}")
            continue

        ann_path = os.path.join(annotations_dir, ann_file)

        # copy files
        shutil.copy(img_path, img_out)
        shutil.copy(ann_path, ann_out)

# ---- Copy files ----
copy_pairs(train_files, output_train_images, output_train_ann)
copy_pairs(val_files, output_val_images, output_val_ann)

print("Split completed successfully!")


Split completed successfully!


# Train Yolo Model

In [4]:
! pip install -U ultralytics

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ------------------- -------------------- 0.5/1.1 MB 284.3 kB/s eta 0:00:02
   ------------------- -------------------- 0.5/1.1 MB 284.3 kB/s eta 0:00:02
   ------------------- -------------------- 0.5/1.1 MB 284.3 kB/s eta 0:00:02
   ------------------- -------------------- 0.5/1.1 MB 284.3 kB/s eta 0:00:02
   ------------------- -------------------- 0.5/1.1 MB 284.3 kB/s e

In [20]:
!yolo detect train data="img/data.yaml" model=yolov8n.pt epochs=50 imgsz=640

Ultralytics 8.3.229  Python-3.9.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=img/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train24, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12

# Predict outpouts

In [9]:
model_path = "E:/Hirusha/data/volley-im-new/Implementation-volley/runs/detect/train18/weights/best.pt"
source_folder = "F:/data/24.08.2025/M3/First half/Final/A-E/"
output_folder = "output_dataset/Ann_A_E"                      # folder to save results
font_scale = 0.4                                   # ↓ smaller label size
thickness = 1                                      # text thickness
box_color = (0, 255, 0)                            # bounding box color (BGR)
text_color = (0, 0, 0)                             # label text color (BGR)

In [11]:
import os
import cv2
import pandas as pd
from ultralytics import YOLO



os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# LOAD MODEL
# -----------------------------
model = YOLO(model_path)

# -----------------------------
# DEFINE COLORS
# -----------------------------
class_colors = {
    "Player_A": (0, 255, 0),     # Green
    "Player_B": (255, 0, 0),   # Blue
    "Judge": (150,100,30),
    "Ball": (0, 0, 255),       # Red
}

font_scale = 0.5
thickness = 1
text_color = (255, 255, 255)

detections_summary = []

# -----------------------------
# RUN INFERENCE IMAGE BY IMAGE
# -----------------------------
for file_name in os.listdir(source_folder):
    if not file_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(source_folder, file_name)
    img = cv2.imread(img_path)

    if img is None:
        print(f"⚠️ Could not read image: {file_name}")
        continue

    # Run YOLO inference
    try:
        results = model.predict(source=img, conf=0.25, save=False, show=False)
    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")
        continue

    # Process results
    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()
        cls = r.boxes.cls.cpu().numpy().astype(int)
        conf = r.boxes.conf.cpu().numpy()

        for box, c, cf in zip(boxes, cls, conf):
            x1, y1, x2, y2 = map(int, box)
            class_name = model.names[c]
            if class_name not in class_colors:
                print(f"⚠️ Skipping unknown class: {class_name}")
                continue

            color = class_colors[class_name]
            label = f"{class_name} {cf:.2f}"

            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
            cv2.rectangle(img, (x1, y1 - h - 5), (x1 + w, y1), color, -1)
            cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, thickness)

            detections_summary.append({
                "image": file_name,
                "class": class_name,
                "confidence": round(float(cf), 3),
                "x1": x1, "y1": y1, "x2": x2, "y2": y2
            })

    # Save annotated image
    save_path = os.path.join(output_folder, file_name)
    cv2.imwrite(save_path, img)
    print(f"✅ Saved annotated: {save_path}")

# -----------------------------
# SAVE DETECTION SUMMARY
# -----------------------------
df = pd.DataFrame(detections_summary)
summary_csv = os.path.join(output_folder, "detections_summary.csv")

if not df.empty:
    df.to_csv(summary_csv, index=False)
    print("\n📊 Class-wise detection count:")
    print(df['class'].value_counts())
    print(f"\n✅ Results saved to: {summary_csv}")
else:
    print("⚠️ No detections found.")


0: 384x640 3 Player_As, 5 Player_Bs, 33.5ms
Speed: 6.2ms preprocess, 33.5ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)
✅ Saved annotated: output_dataset/Ann_A_E\frame_00000.jpg

0: 384x640 3 Player_As, 4 Player_Bs, 3.8ms
Speed: 1.2ms preprocess, 3.8ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)
✅ Saved annotated: output_dataset/Ann_A_E\frame_00001.jpg

0: 384x640 3 Player_As, 4 Player_Bs, 4.0ms
Speed: 1.1ms preprocess, 4.0ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)
✅ Saved annotated: output_dataset/Ann_A_E\frame_00002.jpg

0: 384x640 3 Player_As, 4 Player_Bs, 3.7ms
Speed: 1.1ms preprocess, 3.7ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)
✅ Saved annotated: output_dataset/Ann_A_E\frame_00003.jpg

0: 384x640 3 Player_As, 4 Player_Bs, 4.0ms
Speed: 1.1ms preprocess, 4.0ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)
✅ Saved annotated: output_dataset/Ann_A_E\frame_00004.jpg

0: 384x

KeyboardInterrupt: 